# Predicting degree of utilization from FE mesh

In this notebook we illustrate the usage of the FKM-linear method to compute the degree of utilization
at different nodes of a plate component. Therefore, we proceed the following steps 

    1. Load the FE mesh of a plate with a central hole

    2. Obtain von Mises stresses and stress gradients at the different nodes

    3. Compute the component fatigue limit and the degree of utilization for each node

    4. Visualize the degrees of utilization for all elements included in the FE mesh

In [ ]:
import numpy as np
import pandas as pd
import pyvista as pv

import pylife.stress.equistress
import pylife.mesh.gradient
import pylife.vmap
from scipy.stats import norm

from pylife.strength.fkm_linear.fkm_linear_factors import (
    calc_input_parameters_material,
    calc_input_parameters_stress,
    fatigue_limit_local_chap4,
)

In [ ]:
%store -r rf_dict

Load meash data and add the stress tensor dimensions for the third dimension.

In [ ]:
vm_mesh = pylife.vmap.VMAPImport("plate_with_hole.vmap")
mesh = (vm_mesh.make_mesh('1', 'STATE-2')
               .join_coordinates()
               .join_variable('STRESS_CAUCHY')
               .to_frame())

Calculate von Mises stress

In [ ]:
# the nominal load level in the FEM analysis
mesh['mises'] = mesh.equistress.mises()

Calculate stress gradient on von Mises stress and display mesh data frame

In [ ]:
grad = mesh.gradient.gradient_of('mises')
grad["abs_grad"] = np.linalg.norm(grad, axis = 1)
mesh = mesh.join(grad)
display(mesh)

In the following we initialize the relevant parameters for the FKM linear guideline. These are: 

1. The tensile strength Rm in [MPa], and the Surface roughness Rz in [m*1.0e-6]

2. The relative stress gradient G0 in [1/mm]

3. The material group MatGroupFKM and the material temperature group MatGroupFKM_Temp

4. The GJL material group GJL_Mat, if GJL is used otherwise set to None.

4. The temperature "Temprature" in [°C]

5. The temperature group of the material MatGroupFKM_Temp

6. The diameter "Diameter" in [mm] and the profile "Profile" of the component must be one of {"Rod", "Tube", "Wide sheet", "Rectangle", "Square"}

7. The FKM chapter FKM_chap. Must be one of {'chap4', 'chap5.5'}

8. The Kf method Kf_method for estimating of the fatigue notch factor (Chap. 4.3.1.2).
   Must be one of {'Table', 'Equation'}

9. The support method sup_method representing the support factor calculation after Stieler or FKM2012 or von Mises. Must then be out of  {'Stieler','V90_Mises', 'A90'}

10. The "Condition" variable. Heat treatment condition for fictive width b calculation.  
Must be one of {'Hardened', 'Annealed'}, optinal then set to None

11. The "Finish" variable for surface finish polished (if used, no Rz value necessary).
Must be one of {'polished'}, optional then set to None

12. The "HardProc" variable for choosing the Surface hardening method  
must be one of {'Inductive hardening', 'Flame hardening', 'Case hardening','Carburizing', 'Nitriding', 'Cyaniding', 'Carbonitriding','Deep rolling', 'Shot peening'}, optional then set to None
            
13. The amplitude and meanstress. However, the amplitude will be set to 1 in the following and therefrom the meanstress is inferred using the chosen stress ratio R. 


In [ ]:
# Define settings that are chosen globally, meaning that they do not
# depend on the specific node of an FE-mesh
experiment_settings =  pd.Series({"fkm_chapter": "chap4", "Profile": "Wide sheet","Diameter": 6.0,"Thickness" : 1.0,
                                    "sup_method": "Stieler","Condition": None,"MatGroupFKM": "GJL",
                                    "MatGroupFKM_Temp": "GJL",
                                }
                            )
# Define assessment parameters. This has to be done by the user according to relevant material
# and use case parameters
assessment_parameters = pd.DataFrame({ "Rm" : 910,"Temperature" : 50,"meanstress":0.0,
                                    "GJL_Mat" : "GJL-250","Kf_method" : "Table", "Rz" : 200, "S_Type" : "normal",
                                    "Finish" : None, "HardProc" : None},
                                    index=[0]
                                    )

Now we iterate over all nodes, obtain the gradients and von Mises stresses and compute the component fatigue limit and the degree of utilization for each node.

Finally we visualize the degree of utilization for each element in the FE mesh where we use the average of the degrees of utilization with respect to the nodes in the element.

In [ ]:
# Compute quantities according to FKM-Linear guideline which are related to the material (e.g roughness factor)
# This has to be computed only once (not on each node)
assessment_parameters = calc_input_parameters_material(experiment_settings,assessment_parameters)

# Add the new columns to the original dataframe and rename
mesh = mesh.assign(**assessment_parameters.iloc[0])
mesh = mesh.rename(columns={"abs_grad": "G0", "mises": "amplitude"})

# Apply FKM linear to compute component fatigue limit for each node of the mesh.
# Caution this is a volume mesh demonstrating the usage where the FKM guideline
# only supports the prediction of the component fatigue limit for surface points.
mesh = calc_input_parameters_stress(experiment_settings,mesh)
mesh = fatigue_limit_local_chap4(mesh)
mesh["SDFKM"]

# Choose safety factor and compute degree of utiliaztion for each node
safety_factor = 1
mesh['degree_of_utilization']= (mesh['amplitude'] / mesh['SDFKM']) / safety_factor

# Show mesh data frame
#mesh

In [ ]:
# Visualize the degree of utilization
auslastungs_grad = mesh["degree_of_utilization"]
auslastungs_grad = auslastungs_grad.groupby(['element_id']).mean()
grid = pv.UnstructuredGrid(*mesh.mesh.vtk_data())
plotter = pv.Plotter()
plotter.add_mesh(grid, scalars=auslastungs_grad.to_numpy(), log_scale=True, show_edges=True, cmap='jet')
plotter.add_scalar_bar()
plotter.show()